In [1]:
import json
import math
import random
import re
import requests
import sys
import time
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from datetime import datetime, date, timezone, timedelta
from zoneinfo import ZoneInfo
import psycopg2
from psycopg2.extras import execute_values
sys.path.append('/kaggle/input/datasets/alextatarinoff/tg-functions/')

from telegramfunctions import SendMessageTG

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
DB_HOST = user_secrets.get_secret("DB_HOST")
DB_NAME = user_secrets.get_secret("DB_NAME")
DB_PASSWORD = user_secrets.get_secret("DB_PASSWORD")
DB_USER = user_secrets.get_secret("DB_USER")

TG_CHAT_ID = user_secrets.get_secret("tg_chat_id")
TG_TOKEN = user_secrets.get_secret("tg_token")


DB_PARAMS = {
    "host": DB_HOST,
    "database": DB_NAME,
    "user": DB_USER,
    "password": DB_PASSWORD,
    "port": 5432,
    "sslmode": "require"
}

TABLE_NAME = 'news'

SENDING_OUT_MSGS = True

In [3]:
def parse_sports_date(date_str:str, time_str:str, last_news_month:int, current_parser_year:int)->tuple:
    """
    Конвертирует строки вида '12 августа', '2:15' в datetime.

    :param last_news_month: Номер месяца предыдущей новости в цикле (от 1 до 12)
    :param current_parser_year: Текущий рабочий год парсера (может уменьшаться в процессе)
    :return: (объект datetime, обновленный рабочий год)
    """
    months = {
        'января': 1, 'февраля': 2, 'марта': 3, 'апреля': 4,
        'мая': 5, 'июня': 6, 'июля': 7, 'августа': 8,
        'сентября': 9, 'октября': 10, 'ноября': 11, 'декабря': 12
    }

    try:
        # сплитим на день и месяц, при этом убираются лишние пробелы
        ## от начала и конца, ну вдруг возникнут :)
        date_parts = date_str.split()
        time_str = time_str.split()[0]
        day = int(date_parts[0])
        month_name = date_parts[1]

        # более безопасный способ обращаться в словарь, через .get
        month = months.get(month_name)
        if not month:
            raise ValueError(f"Неизвестное название месяца: {month_name}")

        # чтобы не появлялся nan (он может :)
        if pd.isna(last_news_month):
            last_news_month = None

        # --- ГЛАВНАЯ ЛОГИКА СТЫКА ЛЕТ (МОЯ ИДЕЯ) ---
        # Если мы шли по январю/февралю и вдруг месяц новости стал БОЛЬШЕ
        # (например, декабрь), значит, мы совершили скачок в прошлый год.
        if last_news_month is not None and month > last_news_month:
            current_parser_year -= 1
            print(f"[ПЕРЕХОД ГОДА] Месяц изменился с {last_news_month} на {month}. Год изменён на {current_parser_year}")

        dt_date = date(current_parser_year, month, day)
        dt_time = datetime.strptime(time_str, "%H:%M").time()
        dt = datetime.combine(dt_date, dt_time)
        return dt, current_parser_year

    except Exception as e:
        print(f'Не удалось привести дату и время в формат datetime. Входные значения:')
        print(f'Дата = {date_str}\nВремя = {time_str}\nПосл. месяц = {last_news_month}\nТек. год = {current_parser_year}')
        print(f'ОШИБКА: \n{e}')
        return pd.NaT, current_parser_year

# d = parse_sports_date('  12 сентября', '   2:15', 8, 2026)
# print(d[0], '\nТекущий год', d[1])

In [4]:
# ищем единый блок со всеми новостями на странице
#   и разделяем в блоки по дням

def parse_news_by_day(news_page_tree, last_ids = None):
    '''
    news_page_tree: страница-хаб с новостями
    last_ids - тут передаём последний уже спарсенный нами айдишник - будет служить знаком Стоп, не парсить дальше
                !это проверяется только в одном режиме работы!
    '''

    missed_urls = 0
    reached_saved_news = False
    news_by_day = news_page_tree.find('div', class_='news').find_all('div', class_ ='short-news')


    if not news_by_day:
        print('Не нашли новости, проверьте тэги\классы единого блока новостей и блоков по дням')
    else:
        news_daily = []
        # пока месяца нет, потом вытащим из парсинга даты
        last_news_month = None
        # именно _текущий_ год - потому что в процессе парсинга мы можем уйти
        #   в предыдущий, тогда просто уменьшим переменную, чтобы записывать
        #   дату корректно
        current_parser_year = datetime.now().year

        for data in news_by_day:
            try:
                for day in data.find_all('p'):
                    # начнём с урла, ниже объясняю, почему
                    url = day.find('a', class_='short-text').get('href')
                    # всё просто: обычные футбольные новости на Спортсе начинаются с "/football/1117386157....
                    # если там реклама или видео, урл полный - и нам он не нужен, скипаем эту "новость"
                    # НАШЁЛ ИСКЛЮЧЕНИЕ - МЕДИАФУТБОЛ ЛОЛ
                    if not url.startswith('/'):
                        print('Пропустили урл, не текстовая новость\реклама\другое:', url)
                        missed_urls += 1
                        continue

                    # вторым делом проверяем id на совпадание с уже записанным в БД - не пора ли остановиться?
                    # вырезаем id новости, он нам пригодится
                    #   для более точного поиска последней. сохр. новости - апд. ну вообщето это наш первичный ключ будет ))
                    # апд. 15окт. Внезапно стали попадаться шорт-урлы типа https://www.sports.ru/football/1117401097.html,
                    # и их такая регулярка пропускает... она ждёт "-" после цифер. просто добавил точку в кв. скобках, то есть - или . сработают
                    id_matching = re.search(r"/(\d+)[-.]", url)
                    if id_matching:
                        # достаём только то, что попало в круглые скобки
                            # а вот тут зарылся баг, потому что в news_id мы имели СТРОКУ. А работали дальше как с интом... насильно за intиваем)
                        news_id = int(id_matching.group(1))

                        # проверка, не дошли ли до уже спарсенной новости
                        # ОНА НУЖНА ТОЛЬКО В ОДНОМ РЕЖИМЕ ИЗ ГЛОБ. ПЕРЕМЕННЫХ...

                        if FETCHING_MODE == 'all_from_last_saved' and last_ids == news_id:
                            print('Дошли до уже сохранённой новости, выходим!')
                            reached_saved_news = True
                            #break # косяк - у нас же 2 цикла! из первого выходим, на а дальше в другом дне продолжаем искать то, чего уже там не будет...
                            return (news_daily, reached_saved_news)

                    else:
                        print('Предупреждение: cтруктура ссылки изменилась, ID новости не найден!')
                        print('Входной урл: ', url)
                        news_id = None
                    # вытаскиваем время+дату и превращаем их в pd.datetime
                    ps_date = str(data.b.text)
                    ps_time = day.find('span', class_='time').text
                    pd_datetime, current_parser_year = parse_sports_date(ps_date, ps_time, last_news_month, current_parser_year)
                    last_news_month = pd_datetime.month

                    caption = day.find('a', class_='short-text').text

                    news_daily.append({
                        'datetime': pd_datetime,
                        'caption': caption,
                        'url': url,
                        'news_id': news_id
                    })
            except Exception as e:
                print('Общий блок новостей забрали, но не удалось распарсить конкретные новости')
                print('ОШИБКА: ', e)
                return None

    return (news_daily, reached_saved_news) # только помним, что мы могли не достичь id сохры, потому что их и не было, напр. первый запуск
                                            #   это будем проверять условием if last_ids and (not reached_saved_news) = тогда траблы

In [5]:
def process_last_n_newspages (pages_to_check:int = 1, last_ids = None)->list :
    '''
    Заходит на спортс - новостной хаб по футболу и забирает последние pages_to_check страниц с урлами новостей
    тут же через parse_news_by_day разбирает новость на заголовок, дату, айди и выдаёт это как результат
    last_ids - тут передаём последние (на всяк случай 2-3) уже спарсенные нами айдишники - будет служить знаком Стоп, не парсить дальше
        ну да, они здесь не юзаются, приходится прокидывать "через функцию"
    Возвращаем: список из словарей, в которых - брифы, раскиданные по ключам
    '''

    start_url = 'https://www.sports.ru/football/news/?page='

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    all_news = []
    pages_parsed = 0
    missed_pages = 0
    missed_pages_urls = []  # будем сохранять и прокидывать наружу урлы из хаба, куда не достучались
    reached_saved_news = False
    print('-'*15, 'Запускаем парсинг', '-'*15)
    print(f'РЕЖИМ: {FETCHING_MODE}.', ('Сколько страниц с хаба забираем: ' + str(NUM_PAGES_TO_FETCH)) if FETCHING_MODE == 'latest_n' else '')
    start_time_get = time.perf_counter()

    with requests.Session() as session:
        session.headers.update(headers)

        # в цикле парсим pages_to_check страниц новостей
        for i in range(1, pages_to_check+1):
            final_url = f"{start_url}{i}.html"
            try:
                response = session.get(final_url, timeout=5)
                if response.status_code == 200:
                    print(f"[НАЙДЕНО] Страница {i}: {final_url}")
                    pages_parsed += 1
                else:
                    print(f"Не смогли зайти на страницу: {final_url}, статус: {response.status_code}")
                    missed_pages_urls.append(final_url)
                    missed_pages += 1
                    continue

                tree = BeautifulSoup(response.content, 'html.parser')

                briefs_batch = parse_news_by_day(tree, last_ids)
                all_news.extend(briefs_batch[0])        # добавляем батч новостей-брифов от i-той страницы
                reached_saved_news = briefs_batch[1]    # сохраняем флаг, дошли ли до last known id
                # Тру может быть только в одном режиме: FETCHING_MODE == 'all_from_last_saved' (см. код парсньюсбайдэй), не стал лишнее условие тут ставить
                if reached_saved_news == True:          #   если дошли - выходим, дальше парсить не надо!
                    break

            except requests.exceptions.RequestException as e:
                print(f"Ошибка при запросе {final_url}: {e}")
                time.sleep(10)  # Если сервер упал или сбросил соединение, отдыхаем дольше
                continue

            # делаем рандомную паузу между страницами хаба
            delay = random.uniform(2.5, 8)
            time.sleep(delay)

    end_time_get = time.perf_counter()
    duration_get = end_time_get - start_time_get
    print('-'*49)
    print(f'Заняло времени = {duration_get:.1f} сек, страниц проверено = {pages_to_check}, страниц пропущено: {missed_pages}')

    # по следам реальной работы - надо прокидывать наружу факт missed_pages, чтобы при след. вызове парсера
    #  - ориент. раз в сутки - учесть это и забрать на 1-2 стр. больше
    return (all_news, reached_saved_news, missed_pages_urls)

In [6]:
def json_serial(obj):
    """Превращает даты в строки, чтобы JSON не ругался"""
    if isinstance(obj, datetime):
        return obj.isoformat()  # в строку вида "2026-10-01T08:33:00"
    raise TypeError(f"Тип {type(obj)} не поддерживается")

In [7]:
# =-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=- #
#   -----------  D B   F U N C T I O N S  --------------   #
# =-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=- #

def GetLastIds(db_params, table_name) -> list:
    '''
    db_params - параметры коннекта в ДБ
    table_name - таблица, из которой забираем данные
    Возвращаем: последние 3 айди или None
    '''

    try:
        # 2. Подключаемся к облачной базе данных
        conn = psycopg2.connect(**db_params)
        cursor = conn.cursor()
        print("Успешно подключились к базе данных.")
        sql_query = f'''
        SELECT id, published_at from {table_name}
        ORDER BY published_at DESC
        LIMIT 3
        '''
        cursor.execute(sql_query)
        raw_data = cursor.fetchall()
        print(f'...данные по последним сохранённым новостям получены, посл. id: {raw_data[0][0]}, дата: {raw_data[0][1]}')

    except Exception as e:
        print('Не удалось подключиться к БД', table_name, 'Ошибка', e)
        return None

    finally:
        # Обязательно закрываем соединение
        if 'cursor' in locals(): cursor.close()
        if 'conn' in locals(): conn.close()
        print("Соединение с БД закрыто.")

    return raw_data


def SavingBriefs_DB(briefs, db_params, table_name):

     # способ от ии, ну ок
    data_to_insert = [
        (
            news.get('news_id'),
            news.get('caption'),
            news.get('text'),
            news.get('tags'),
            news.get('url'),
            # Если это datetime объект, psycopg2 сам правильно переведет его в формат Postgres!
            news.get('datetime'),
            'sports.ru',
        )
        for news in briefs if news.get('news_id')  # доп. перестраховка, но None в айди сюда доходить не должны
    ]
    try:
        # подключаемся к облачной базе данных
        conn = psycopg2.connect(**db_params)
        cursor = conn.cursor()
        print(" ...успешно подключились к базе данных")

        # массовая вставка через execute_values
        # если Id уже есть - пропускаем (заголовки временами меняют и даже статьи поправляют, но в этом случае пересечение по id будет минимальным, макс 1-2 id)
        query = f"""
        INSERT INTO {table_name} (id, title, text, tags, url, published_at, source)
        VALUES %s
        ON CONFLICT (id) DO NOTHING
        RETURNING id
        """
        inserted_ids = execute_values(cursor, query, data_to_insert, fetch=True)
        real_inserted = len(inserted_ids)
        print(f" ...новостей получено: {len(data_to_insert)}, в базу данных успешно загружено: {real_inserted}.")
        conn.commit()

    except Exception as e:
        print(f"Произошла ошибка: {e}")

    finally:
        # обязательно закрываем соединение
        if 'cursor' in locals(): cursor.close()
        if 'conn' in locals(): conn.close()
        print("Соединение с БД закрыто.")


def SavingBriefs_JSON(briefs, filename):
    try:
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(briefs, f, indent=4, default=json_serial)
        print(f"Брифы успешно сохранены локально в файл {filename}")
    except Exception as e:
        print('Не удалось сохранить брифы как json-файл:', e)
        return False
    return True

In [8]:
def GetNeededNumPages(lastdate: datetime) -> int:
    '''
    калькулируем, сколько страниц с хаба "брифов" нам нужно забрать, чтобы
    точно дойти до уже лежащей в БД последней новости
    '''
    # текущее время с таймзоной +3
    now = datetime.now(tz=timezone(timedelta(hours=3)))

    # timedelta - простое вычитание, кул
    delta = now.replace(tzinfo=None) - lastdate
    # Разницу в днях окр. "вверх", чтобы не было 0 страниц
    days_diff = math.ceil(delta.total_seconds() / 86400)
    # на спортсе за день накатывают 1,5-2 страницы на новостном хабе
    #  добавляем х2 для верности
    number_of_pages_needed = math.ceil(days_diff*2)

    return number_of_pages_needed

In [10]:
# =-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
#   -------------  M A I N   C Y C L E  ----------------
# =-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-

def main(mode):
    print(f'{"%"*10} |{datetime.now(ZoneInfo("Europe/Moscow")).strftime("%Y-%m-%d %H:%M:%S")}| Работает скрипт по парсингу и сохранению новостных брифов {"%"*10}\n')
    last_ids = GetLastIds(DB_PARAMS, TABLE_NAME)
    
    if mode == 'all':
        num_pages_to_process = 20
    elif mode == 'all_from_last_saved':
        num_pages_to_process = GetNeededNumPages(last_ids[0][1])
    else:
        num_pages_to_process = NUM_PAGES_TO_FETCH  # это режим latest_n_pages

    # первое главное действо - получаем все брифы к себе...
    all_briefs = process_last_n_newspages(num_pages_to_process, last_ids[0][0])
    
    if all_briefs[0]:   # ...ващет надо проверить, а в новостях-то не пусто?
        reached_saved_news = all_briefs[1]
        print('Достигли ли последней сохранённой новости:', reached_saved_news)
        # если были missed_pages с парсинга хаба брифов - надо отметить этот момент...
        if all_briefs[2]:
        # пока просто принтом
            print(f'ВНИМАНИЕ: На этапе парсинга новостного хаба не удалось подключиться к след. страницам: {all_briefs[2]}.')
    
        try:
            print('Готовимся записать спарсенные брифы в БД...')
            SavingBriefs_DB(all_briefs[0], DB_PARAMS, TABLE_NAME)
            msg_out = f'☑ Скрипт по парсингу\сохранению новостных брифов отработал штатно, забрали и сохранили {len(all_briefs[0])} шт.'
        except Exception as e:
            print('Не получилось записать брифы в БД, пытаемся записать в JSON. Ошибка:', e)
            try:
                SavingBriefs_JSON(all_briefs[0], f'newsbriefs_{datetime.now().strftime("%Y-%m-%d_%H-%M")}.json')
                msg_out = f'⚠ Скрипт по парсингу\сохранению новостных брифов отработал не совсем штатно, забрали и сохранили ТОЛЬКО в JSON (не в БД) {len(all_briefs[0])} шт.'
            except Exception as e:
                print('И в JSON не получилось записать брифы, что за напасть! Ошибка:', e)
                msg_out = f'☒ Скрипт по парсингу новостных брифов отработал штатно, но сохранить результаты не удалось ни в БД, ни локально в JSON'
    else:
        print('\nНовых данных не обнаружено, записывать в БД нечего.')
        msg_out = '⚠ Скрипт по парсингу\сохранению новостных брифов завершился: либо новостей не было, либо произошли неполадки'

    if SENDING_OUT_MSGS:
        msg_is_sent = SendMessageTG(msg_out, TG_TOKEN, TG_CHAT_ID)    
        if not msg_is_sent:
            print('БЫВАЕТ: результат запуска скрипта не удалось отправить в Телеграм')
    
    print('%'*100)

In [11]:
# режим, в котором забираем новости
# latest_n_pages - полностью последние n = NUM_PAGES_TO_FETCH
# all_from_last_saved - все новости с момента последней записи в БД до сейчас
# all - парсить всё :) ну ок, это условно, реально поставил 1000 страниц
#     и надо учитывать, что на спортсе макс 20 страниц на хабе, то есть
#     этот режим надо отслеживать в процессе фетча, впрочем, как и all_unfetched.
#   апд.9.10: хм, а как его отследишь, на спортсе 21 страница - нет, но идёт переход на 1-ю страницу...
#           то есть можно тысячи раз парсить те же самые новости...
#            ок, потом над этим подумаем, сейчас оставил максимум 20 страниц на хабе

FETCHING_MODE = 'all_from_last_saved'
# FETCHING_MODE = 'latest_n_pages'
NUM_PAGES_TO_FETCH = 1  # 2 страницы = 200 новостных урлов


if __name__ == '__main__':
    main(FETCHING_MODE)

%%%%%%%%%%%% |2026-10-22 03:33:49| Работает скрипт по парсингу и сохранению новостных брифов %%%%%%%%%%%%

Успешно подключились к базе данных.
...данные по последним сохранённым новостям получены, посл. id: 1117410042, дата: 2026-10-22 02:01:00
Соединение с БД закрыто.
--------------- Запускаем парсинг ---------------
РЕЖИМ: all_from_last_saved. 
[НАЙДЕНО] Страница 1: https://www.sports.ru/football/news/?page=1.html
Пропустили урл, не текстовая новость\реклама\другое: https://video.sports.ru/video/2RQnaFk_1B_URmb5M7JOG3l5645bXZJoqSTkgd7jUWY
Дошли до уже сохранённой новости, выходим!
-------------------------------------------------
Заняло времени = 4.4 сек, страниц проверено = 2, страниц пропущено: 0

Новых данных не обнаружено, записывать в БД нечего
Отправка запроса в Telegram...
Статус ответа от сервера: 200. Бот успешно прислал сообщение в Telegram
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
